~~~
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
~~~

# MedGemma 1.5를 사용한 컴퓨터 단층촬영(CT) 영상 분석

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/high_dimensional_ct_model_garden.ipynb">
      <img alt="Google Colab logo" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br> Google Colab에서 실행
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogle-Health%2Fmedgemma%2Fmain%2Fnotebooks%2Fhigh_dimensional_ct_model_garden.ipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Colab Enterprise에서 실행
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-health/medgemma/blob/main/notebooks/high_dimensional_ct_model_garden.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> GitHub에서 보기
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://huggingface.co/collections/google/medgemma-release-680aade845f90bec6a3f60c4">
      <img alt="Hugging Face logo" src="https://huggingface.co/front/assets/huggingface_logo-noborder.svg" width="32px"><br> View on Hugging Face
    </a>
  </td>
</tr></tbody></table>

이 노트북은 Vertex AI에서 실행되는 MedGemma 1.5에 프롬프트를 보내기 위해 [컴퓨터 단층촬영(CT)](https://www.nibib.nih.gov/science-education/science-topics/computed-tomography-ct) 영상의 3D 표현을 사용하는 방법을 보여줍니다. 이 노트북은 MedGemma 1.5의 기본 기능을 보여주기 위한 교육 목적으로만 제공됩니다. 완성되거나 승인된 제품이 아니며, 어떠한 질병이나 상태에 대한 진단 또는 치료를 제안하기 위한 것이 아니므로 의학적 조언으로 사용되어서는 안 됩니다. 자세한 내용은 [HAI-DEF 이용 약관](https://developers.google.com/health-ai-developer-foundations/terms)을 참조하세요.

Vertex AI를 사용하면 모델을 쉽게 서빙(serve)하고 통신할 수 있습니다. [Vertex AI](https://cloud.google.com/vertex-ai/docs/start/introduction-unified-platform)에 대해 자세히 알아보세요.

### Costs

이 튜토리얼에서는 요금이 청구될 수 있는 다음의 Google Cloud 구성 요소를 사용합니다:

* Vertex AI

[Vertex AI 가격](https://cloud.google.com/vertex-ai/pricing)에 대해 알아보고, [가격 계산기](https://cloud.google.com/products/calculator/)를 사용하여 예상 사용량에 기반한 예상 비용을 산출해 보세요.


In [ ]:
# @title pydicom Python 라이브러리 설치

%%capture
! pip install pydicom

In [ ]:
# @title Google Cloud 인증 (Google Colab에서만 필요)
import os
import sys

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE"

if google_colab:
    from google.colab import auth
    # 사용자 계정으로 로그인하고 액세스를 승인하라는
    # 팝업이 나타납니다.
    auth.authenticate_user()

## Imaging Data Commons (IDC) CT 이미지 메타데이터 검색

[Imaging Data Commons (IDC)](https://datacommons.cancer.gov/repository/imaging-data-commons#)는 암 영상을 위해 공개적으로 사용할 수 있는, 가장 큰 비식별화된 저장소 중 하나입니다. 이 저장소는 [미국 보건복지부(U.S. Department of Health and Human Services)](https://www.hhs.gov/) 소속 [국립보건원(NIH)](https://www.nih.gov/) 산하 기관인 [국립암연구소(NCI)](https://www.cancer.gov/)의 지원을 받습니다. IDC에는 모든 주요 의료 영상 양식의 이미지가 포함되어 있습니다. 이미지는 [DICOM](https://www.dicomstandard.org/)으로 아카이브 내에 저장됩니다. 이미지 및 관련 메타데이터는 [IDC 웹사이트](https://portal.imaging.datacommons.cancer.gov/explore/) 또는 [BigQuery](https://cloud.google.com/healthcare-api/docs/resources/public-datasets/idc)를 통해 검색 및 시각화할 수 있으며, DICOMweb([IDC 튜토리얼](https://learn.canceridc.dev/data/downloading-data/dicomweb-access))을 사용하여 액세스할 수 있습니다. 이 노트북의 목적을 위해 필수 이미지를 Google DICOM 스토어에 미러링했습니다.

DICOM은 CT 스캐너에서 생성되는 의료 영상 형식입니다.
[DICOM](https://www.dicomstandard.org/) 이미지는 세 가지 UID(Study Instance UID, Series Instance UID, SOP Instance UID)로 고유하게 식별됩니다. 이 노트북은 CT 스캔과 관련된 모든 이미지를 다운로드하여 IDC에서 CT 이미지를 검색합니다. 개념적으로 Study Instance UID는 환자 검사의 결과로 획득하거나 생성된 모든 이미지를 식별하는 UID로 생각할 수 있습니다. 검사의 일환으로 획득된 각 의료 이미지(예: CT 획득)는 고유한 Series Instance UID로 식별됩니다. 반면 취득의 일부로 획득하거나 생성된 각 이미지는 고유한 SOP Instance UID로 식별됩니다.

CT 이미징은 일반적으로 2D 이미지(슬라이스)의 순서가 지정된 모음으로 표현됩니다. 각 슬라이스는 이미지로 촬영된 신체의 축방향 단면(볼륨)을 설명하는 2D 이미지로 표시됩니다.



In [ ]:
# @title 시리즈 내 DICOM 인스턴스에 대한 메타데이터 읽기
from typing import Sequence

import google.auth
import google.auth.transport
import pydicom
import requests


def get_auth_token() -> str:
  auth_credentials = google.auth.default(['https://www.googleapis.com/auth/cloud-healthcare'])[0]
  auth_credentials.refresh(google.auth.transport.requests.Request())
  return auth_credentials.token


def get_sop_instance_dicomweb_urls(study_instance_uid: str, series_instance_uid: str) -> Sequence[str]:
  # 이미징용 DICOM 인스턴스 메타데이터 읽기
  # 스토어에서 읽기 위한 oauth 자격 증명 얻기
  headers = {"Authorization" : f"Bearer {get_auth_token()}"}
  dicomweb_series_url = f"https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/{study_instance_uid}/series/{series_instance_uid}"
  metadata = requests.get(f"{dicomweb_series_url}/instances", headers=headers).json()
  dicom_instances = [pydicom.Dataset.from_json(i) for i in metadata]
  # 인스턴스 번호별로 인스턴스 메타데이터 정렬
  dicom_instances = sorted(dicom_instances, key=lambda i: int(i.InstanceNumber))

  # CT 볼륨에서 85개 조각을 균일하게 샘플링
  MAX_SLICE = 85
  if len(dicom_instances) > MAX_SLICE:
    dicom_instances = [dicom_instances[int(round(i /(MAX_SLICE) * (len(dicom_instances)-1)))] for i in range(1, MAX_SLICE + 1)]
  return [f'{dicomweb_series_url}/instances/{dcm.SOPInstanceUID}' for dcm in dicom_instances]

In [ ]:
# @title ## 채팅 완료(Chat Completion) 형식으로 MedGemma 1.5 프롬프트 구성

# 이 노트북은 IDC(Imaging Data Commons) 아카이브에서 호스팅되는 이미지를 사용합니다.
# 이 노트북은 암이미징 아카이브(TCIA, The Cancer Imaging Archive)의 데이터를 활용합니다.
# https://www.cancerimagingarchive.net/collection/colorectal-liver-metastases/
# Liver CRLM - CT 1026
study_instance_uid = "1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436"
series_instance_uid = "1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220"

dicom_source = get_sop_instance_dicomweb_urls(study_instance_uid, series_instance_uid)

instruction = ("You are an instructor teaching medical students. You are "
               "analyzing a contiguous block of CT slices from the center of "
               "the abdomen. Please review the slices provided below "
               "carefully.")
content = [{"type": "text", "text": instruction}]
content.append({"type": "image_dicom", "image_dicom": {"dicom_source": dicom_source, 'access_credential': get_auth_token()}})
query_text = ("\n\nBased on the visual evidence in the slices provided above, "
              "is this image a good teaching example of liver pathology? "
              "Comment on hypodense lesions or other hepatic irregularities. "
              "Do not comment on findings outside the liver. Please provide "
              "your reasoning and conclude with a 'Final Answer: yes' or "
              "'Final Answer: no'.")
content.append({"type": "text", "text": query_text})

instance = {
    "@requestFormat": "chatCompletions",
    "messages": [{"role": "user", "content": content}],
    "max_tokens": 500,
    "temperature": 0
}

In [ ]:
# @title ## MedGemma 1.5 프롬프트 표시
import json
from IPython.display import display, Markdown

def mask_bearer_token(obj):
  """Masks bearer token."""
  if isinstance(obj, dict):
    return {k: mask_bearer_token(v) if k != 'access_credential' else 'TOKEN' for k, v in obj.items()}
  elif isinstance(obj, list):
    return [mask_bearer_token(elem) for elem in obj]
  return obj

txt = json.dumps(mask_bearer_token(instance), indent=4, sort_keys=True)
display(Markdown(f'```json\n{txt}'))

```json
{
    "@requestFormat": "chatCompletions",
    "max_tokens": 500,
    "messages": [
        {
            "content": [
                {
                    "text": "You are an instructor teaching medical students. You are analyzing a contiguous block of CT slices from the center of the abdomen. Please review the slices provided below carefully.",
                    "type": "text"
                },
                {
                    "image_dicom": {
                        "access_credential": "TOKEN",
                        "dicom_source": [
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.475273662596035314302684328529",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.231142193158026555203864746037",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.302297530008610262481602047087",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.190960328130437367326276502376",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.214089864075573237476119770626",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.174332313835818207466803310411",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.180537612945505818606587783969",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.405102839118124449252609177118",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.266761757042739995245700278237",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.227436103971867794261713320202",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.600064396808909835377922556388",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.186939469153882962728793601203",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.170998679375101828913637577724",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.244694012748215200319554940658",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.852414061205359600465498795985",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.209262910331802821465596992536",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.327185406336369507258611871750",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.179687663422977549475519907111",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.181474902463086566230854381436",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.233514649674291726915754979900",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.361532762406708143738372646850",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.830613239463755152804216145996",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.301552068461004453076029701451",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.222517415400664504186695471282",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.313372316268460883375154695008",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.189189473919011968547669685121",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.225044713289996559967925925833",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.755399102190683505013136070851",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.546156832809619538552768705199",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.315161309265302356758091759152",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.110588217517785610957724565828",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.303925255271219615595537739043",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.332089664826268215512300446108",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.119909471920231057343036958985",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.216665802898138558389872772396",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.872011087736177264730437480446",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.837829465261434842508182774331",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.323865487309226783908225736915",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.302572411786247455876129428847",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.219934395340248254875845941851",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.222734114401353419027416481676",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.275630861214679094027634786561",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.266347882205983951341441440857",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.131581330498440215128617078937",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.265806801324990089044925893194",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.249591715516516680330434213730",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.155964817805406786372373755702",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.162655284282573812757193808972",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.306253407785342121139904867166",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.149199060934943003148353214800",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.190402308117504884628191661084",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.145397390350809400014616174867",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.320119403924772097576500249015",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.161655152316742359799872016786",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.820448157649521320746231934821",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.283565907139287625834050240706",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.157231446533457413659128601762",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.107101751675260887276054795590",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.198186304108503975604234239454",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.109081856441304431617859035749",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.215616020765651148269769057924",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.149081807955788195839266371066",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.109724447573146232220262090401",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.129668256345963751797706445020",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.506095083286473705236474997506",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.172291884077510187156668669018",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.117052723010670105722132967581",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.108399401097333823351482255825",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.414296426287777112097850828389",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.228562660281490329234551679302",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.161194399437240717202306532389",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.168771884587837583168007593325",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.686825290080668417559582848168",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.477441187799385355060835584898",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.884193210431343619286615792179",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.307213070982087854088686341690",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.138300959540467713707195195327",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.702927050896958784914824175980",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.376835262753192890804744483788",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.325494907912019801920046789482",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.588215090796023621376225975050",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.327342093919148238560554060312",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.211426399929246327923268702156",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.142106182050391254108946189706",
                            "https://healthcare.googleapis.com/v1/projects/hai-cd3-foundations/locations/us-central1/datasets/public/dicomStores/test-images/dicomWeb/studies/1.3.6.1.4.1.14519.5.2.1.9203.8273.982856921320609617394372605436/series/1.3.6.1.4.1.14519.5.2.1.9203.8273.275179554444442893192427753220/instances/1.3.6.1.4.1.14519.5.2.1.9203.8273.770073332760280770187004140942"
                        ]
                    },
                    "type": "image_dicom"
                },
                {
                    "text": "\n\nBased on the visual evidence in the slices provided above, is this image a good teaching example of liver pathology? Comment on hypodense lesions or other hepatic irregularities. Do not comment on findings outside the liver. Please provide your reasoning and conclude with a 'Final Answer: yes' or 'Final Answer: no'.",
                    "type": "text"
                }
            ],
            "role": "user"
        }
    ],
    "temperature": 0
}

In [ ]:
# @title ## Google Cloud 환경 및 MedGemma 1.5 Vertex AI 엔드포인트 설정

# @markdown #### 전제 조건
# @markdown
# @markdown 1. 프로젝트에 대해 [결제가 활성화되어 있는지](https://cloud.google.com/billing/docs/how-to/modify-project) 확인하세요.
# @markdown
# @markdown 2. Compute Engine API가 활성화되어 있는지 아니면 API를 활성화할 [Service Usage 관리자](https://cloud.google.com/iam/docs/understanding-roles#serviceusage.serviceUsageAdmin) (`roles/serviceusage.serviceUsageAdmin`) 역할이 있는지 확인하세요.
# @markdown
# @markdown 이 섹션은 기본 Google Cloud 프로젝트를 설정하고 Compute Engine API를 활성화(아직 활성화되지 않은 경우)하고 Vertex AI API를 초기화합니다.

import os
from google.cloud import aiplatform

Google_Cloud_Project = ""  # @param {type: "string", placeholder:"e.g. MyProject"}

# @markdown [온라인 추론(online inferences)](https://cloud.google.com/vertex-ai/docs/predictions/get-online-predictions)을 얻으려면 Model Garden에서 배포된 MedGemma 1.5 [Vertex AI 엔드포인트](https://cloud.google.com/vertex-ai/docs/general/deployment)가 필요합니다. 아직 배포하지 않은 경우 [MedGemma 모델 카드](https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/medgemma)로 이동하여 **"Deploy model"**을 클릭하여 모델을 배포하세요.
# @markdown
# @markdown **참고:**
# @markdown - 이 노트북은 MedGemma 1.5와 함께 사용하도록 만들어졌습니다. 최소 85개의 이미지 제한 형식으로 재설정된 MedGemma 1.5 엔드포인트를 배포하고 사용하는지 확인하세요.(Model Garden **"Deploy options"** 패널에서 **"Deployment Settings" > "Serving spec"** 아래 이미지 제한을 확인합니다).
# @markdown - 이 노트북에는 [전용(dedicated) Vertex AI 엔드포인트](https://cloud.google.com/vertex-ai/docs/predictions/choose-endpoint-type)가 필요합니다. 전용 엔드포인트는 Model Garden 배포에 대한 기본 엔드포인트 유형입니다.

# @markdown 이 섹션은 온라인 추론을 사용하기 위해 Model Garden에서 배포한 Vertex AI 엔드포인트 리소스를 가져옵니다.
# @markdown
# @markdown 아래에 엔드포인트 ID와 리전을 채우세요. 배포된 엔드포인트는 [Vertex AI endpoints 페이지](https://console.cloud.google.com/vertex-ai/online-prediction/endpoints)에서 찾을 수 있습니다.

ENDPOINT_ID = ""  # @param {type: "string", placeholder:"e.g. 123456789"}
ENDPOINT_REGION = ""  # @param {type: "string", placeholder:"e.g. us-central1"}

os.environ["CLOUDSDK_CORE_PROJECT"] = Google_Cloud_Project
os.environ["GOOGLE_CLOUD_PROJECT"] = Google_Cloud_Project
os.environ["GOOGLE_CLOUD_REGION"] = ENDPOINT_REGION

# (아직 없다면) Compute Engine API 활성화.
print("Enabling Compute Engine API.")
! gcloud services enable compute.googleapis.com

# Vertex AI API 초기화.
print("Initializing Vertex AI API.")
aiplatform.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
                location=os.environ["GOOGLE_CLOUD_REGION"],
                api_transport="rest")

endpoint = aiplatform.Endpoint(
    endpoint_name=ENDPOINT_ID,
    project=Google_Cloud_Project,
    location=ENDPOINT_REGION,
)

# 엔드포인트 이름을 사용하여 적절한 모델 변형을 사용하고 있는지 확인합니다.
# 이러한 확인은 Model Garden 배포 설정의 기본 엔드포인트 이름을 기반으로
# 이루어집니다.
ENDPOINT_NAME = endpoint.display_name
# "1.5"("1_5" 형식)가 엔드포인트 이름에 포함되어 있는지 확인합니다.
if "1_5" not in ENDPOINT_NAME:
    raise ValueError(
        "The examples in this notebook are intended to be used with MedGemma "
        "1.5. Please deploy and use an endpoint with the MedGemma 1.5 model."
    )

Enabling Compute Engine API.


To take a quick anonymous survey, run:
  $ gcloud survey

Initializing Vertex AI API.


In [ ]:
# @title ## MedGemma 1.5 호출(Call) 및 예측 결과 반환

import json
from IPython.display import display, Markdown

response = endpoint.raw_predict(
    body=json.dumps(instance).encode('utf-8'), use_dedicated_endpoint=True,
    headers={'Content-Type': 'application/json'}
)
response.raise_for_status()
medgemma_response = response.json()
try:
  medgemma_response = medgemma_response["choices"][0]["message"]["content"]
  display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{medgemma_response}\n\n---"))
except KeyError:
  print(medgemma_response["error"])


---

**[ MedGemma ]**

Based on the provided CT image, there is a hypodense lesion in the right hepatic lobe. This lesion appears somewhat ill-defined and has a slightly heterogeneous appearance.

The liver parenchyma itself appears generally normal in terms of overall density and contour. There are no obvious signs of cirrhosis, fatty infiltration, or other gross parenchymal abnormalities.

Therefore, this image could serve as a teaching example demonstrating a focal liver lesion.

Final Answer: yes

---

## 다음 단계

다른 [노트북](https://github.com/google-health/medgemma/blob/main/notebooks)을 탐색하여 이 모델로 또 무엇을 할 수 있는지 알아보세요.
